
SPDX-FileCopyrightText: Copyright (c) 1993-2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License"); you may not use

this file except in compliance with the License. You may obtain a copy of the License at



http://www.apache.org/licenses/LICENSE-2.0



Unless required by applicable law or agreed to in writing, software

distributed under the License is distributed on an "AS IS" BASIS,

WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.

See the License for the specific language governing permissions and

limitations under the License.



# TensorRT 入门：加速你的深度学习推理

欢迎来到你的第一个 TensorRT 教程！在本笔记本中，你将学习如何：
1. 加载预训练的 ONNX 格式 EfficientNet 模型
2. 将其转换为 TensorRT 引擎以实现更快的推理
3. 运行推理并亲身体验加速效果
4. 对真实图像进行预测

## 理解 ONNX：通用模型格式

ONNX（Open Neural Network Exchange）是一种用于表示深度学习模型的标准格式。你可以将其视为不同深度学习框架都能理解的通用语言。其重要性体现在：

- **框架无关性**：在 PyTorch、TensorFlow 或其他框架中训练的模型均可导出为 ONNX
- **互操作性**：ONNX 模型可被导入各种推理引擎和框架
- **生产就绪**：ONNX 在生产环境的模型部署中被广泛采用

### ONNX 到 TensorRT 的工作流程

TensorRT 是 NVIDIA 的深度学习推理优化器，支持从 ONNX 导入模型。这使其成为部署流水线中的强大工具：

```
你的框架（PyTorch/TF/等） → ONNX → TensorRT ===> 优化后的推理
```

此工作流程的强大之处在于：
1. 你可以使用任何偏好的框架训练模型
2. 将其导出为 ONNX（一次性转换）
3. 使用 TensorRT 针对 NVIDIA GPU 进行优化
4. 在生产环境中获得显著的速度提升

## 前置条件

在开始之前，请确保你已具备：
- 支持 CUDA 的 NVIDIA GPU
- 已安装 Python 3.10+
- 具备深度学习和推理的基础知识

让我们首先安装并导入所需的软件包：

In [1]:
%pip install tensorrt cuda-python pillow onnxruntime
import tensorrt as trt
from cuda.bindings import runtime as cudart
from PIL import Image
import numpy as np
from pathlib import Path
import time
from typing import Optional, Union, Tuple

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [2]:
root = Path.cwd()

In [3]:
# define a function to download files

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def download_file(url: str, output_path: Union[str, Path]):
    """Download a file with retry mechanism."""
    session = requests.Session()
    retry = Retry(total=10, backoff_factor=1)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    response = session.get(url, verify=False, timeout=30)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as f:
        f.write(response.content)

## 步骤 1：下载预训练模型

我们将以 EfficientNet-B0 为例，这是一个广受欢迎且高效的图像分类模型。

### 理解 ONNX 模型结构

一个 ONNX 模型包含：
- 模型架构（层、连接）
- 权重与偏置
- 输入/输出规范
- 模型元数据，与其他模型表示形式一致。

这种标准化格式使得模型在不同框架和推理引擎之间迁移变得便捷。

In [4]:
download_file("https://github.com/onnx/models/raw/refs/heads/main/Computer_Vision/efficientnet_b0_Opset17_timm/efficientnet_b0_Opset17.onnx", root / "efficientnet-b0.onnx")
assert (root / "efficientnet-b0.onnx").exists(), "Model file not found. Please check if the download was successful."

/opt/conda/envs/cp312/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/conda/envs/cp312/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'media.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 步骤 2：将 ONNX 转换为 TensorRT 引擎

这就是神奇之处！我们将把 ONNX 模型转换为 TensorRT 引擎。该引擎针对你的特定 GPU 进行了优化，运行速度将远快于原始模型。

### 转换流程

1. **加载 ONNX 模型**：TensorRT 读取 ONNX 文件并解析模型结构
2. **优化**：TensorRT 执行多项优化操作：
   - 层融合
   - 内存优化
   - 精度校准
3. **生成引擎**：创建高度优化的推理引擎

生成的引擎专用于你的 GPU，其运行速度将显著快于原始 ONNX 模型。

In [5]:
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED))

# Bind the TensorRT network to the parser so that the parser can update the network later accordingly
parser = trt.OnnxParser(network, logger)

onnx_path = root / "efficientnet-b0.onnx"
print(f'Parsing ONNX model at {onnx_path}...')
with open(onnx_path, "rb") as model:
    parser.parse(model.read())
print('Parsing ONNX model... done')

Parsing ONNX model at /root/cys/PROJECT/00-COMMON/DEMO/02-TensorRT/samples/python/refactored/1_run_onnx_with_tensorrt/efficientnet-b0.onnx...
Parsing ONNX model... done


Now that we have the TensorRT `INetworkDefinition`, we can start building the engine

In [6]:
config = builder.create_builder_config()

# TensorRT needs memory for layer operations and intermediate activations during inference
# Setting a memory limit helps control resource usage and prevents out-of-memory errors
config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE, 1 << 30
) # 1GB

print('Starting to build engine. This might take several minutes depending on the hardware...')
engine = builder.build_serialized_network(network, config)
assert engine is not None, 'Engine build failed'

engine_path = root / "efficientnet-b0.plan"
with open(engine_path, 'wb') as f:
    f.write(engine)

print("TensorRT engine created successfully!")

Starting to build engine. This might take several minutes depending on the hardware...
TensorRT engine created successfully!


## 可选：使用可编辑时序缓存

TensorRT引擎在不同构建版本间可能存在差异，因为内核选择基于运行时性能测量。硬件状态（GPU利用率、温度、系统负载）会影响内核的选择，因为不同内核在各种场景下的表现可能优劣互现。

为确保构建的一致性，TensorRT提供了可编辑时序缓存，其具备以下功能：
- 存储中间优化结果
- 实现确定性引擎构建
- 加速后续构建过程，无需再次测量每个算子的内核执行时间

In [7]:
def build_engine_with_cache(onnx_path: Union[str, Path], timing_cache: Optional[trt.ITimingCache]):
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED))
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, 'rb') as model:
        parser.parse(model.read())
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

    # Enable editable timing cache
    config.set_flag(trt.BuilderFlag.EDITABLE_TIMING_CACHE)

    # Create timing cache if not provided
    if not timing_cache:
        timing_cache = config.create_timing_cache(bytes())
    config.set_timing_cache(timing_cache, True)

    # Build engine
    print('Start building engine...')
    tik = time.time()
    engine = builder.build_serialized_network(network, config)
    tok = time.time()

    print(f'Engine build cost {tok - tik}ms')
    return engine, timing_cache

# First build (creates cache)
engine1, timing_cache = build_engine_with_cache(onnx_path, None)
print("First build completed with cache creation")

# Second build (uses cache)
engine2, timing_cache = build_engine_with_cache(onnx_path, timing_cache)
print("Second build completed with cache creation")

is_identical = np.array_equal(
    np.frombuffer(engine1, dtype=np.uint8),
    np.frombuffer(engine2, dtype=np.uint8))
print(f'Is engine identical: {is_identical}')

Start building engine...
Engine build cost 302.34697699546814ms
First build completed with cache creation
Start building engine...


KeyboardInterrupt: 

## 步骤 3：运行推理并比较性能

现在让我们见证 TensorRT 的真正威力！我们将：
1. 分别使用 ONNX 和 TensorRT 运行推理
2. 比较它们的性能表现
3. 观察 TensorRT 带来的加速效果

### 理解性能差异的原因

这种加速效果源于多项优化技术：
- 层融合：将多个运算操作合并为单一操作
- 内存优化：改进内存访问模式
- 精度优化：为各层选用最优精度
- CUDA 优化：绕过框架开销实现直接 GPU 执行

In [ ]:
def load_and_preprocess_image(image_path: Union[str, Path], input_size: Tuple[int, int] = (224, 224)):
    img = Image.open(image_path)
    img = img.resize(input_size)
    img = np.array(img).astype(np.float32)
    img = img / 255.0  # Normalize from [0, 255] to [0, 1]
    img = np.transpose(img, (2, 0, 1))  # HWC to CHW
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

def check_cuda_error(error):
    if isinstance(error, tuple):
        error = error[0]
    if error != cudart.cudaError_t.cudaSuccess:
        error_name = cudart.cudaGetErrorName(error)[1]
        error_string = cudart.cudaGetErrorString(error)[1]
        raise RuntimeError(f"CUDA Error: {error_name} ({error_string})")

def run_inference_trt(engine: trt.ICudaEngine, input_data: np.ndarray):
    # Create execution context - this stores the device memory allocations
    # and bindings needed for inference
    context = engine.create_execution_context()

    # Initialize lists to store input/output information and GPU memory allocations
    inputs = []
    outputs = []
    allocations = []

    # Iterate through all input/output tensors to set up memory and bindings
    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        # Check if this tensor is an input or output
        is_input = engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT
        # Get tensor datatype and shape information
        dtype = engine.get_tensor_dtype(name)
        shape = engine.get_tensor_shape(name)

        # Calculate required memory size for this tensor
        size = np.dtype(trt.nptype(dtype)).itemsize
        for s in shape:
            size *= s

        # Allocate GPU memory for this tensor
        err, allocation = cudart.cudaMalloc(size)
        check_cuda_error(err)

        # Store tensor information in a dictionary for easy access
        binding = {
            "index": i,
            "name": name,
            "dtype": np.dtype(trt.nptype(dtype)),
            "shape": list(shape),
            "allocation": allocation,
            "size": size,
        }

        # Keep track of all allocations and sort tensors into inputs/outputs
        allocations.append(allocation)
        if is_input:
            inputs.append(binding)
        else:
            outputs.append(binding)

    # Ensure input data is contiguous in memory for efficient GPU transfer
    input_data = np.ascontiguousarray(input_data)

    # Copy input data from host (CPU) to device (GPU)
    err = cudart.cudaMemcpy(
        inputs[0]["allocation"],
        input_data.ctypes.data,
        inputs[0]["size"],
        cudart.cudaMemcpyKind.cudaMemcpyHostToDevice,
    )
    check_cuda_error(err)

    # Set tensor addresses for all tensors
    for i in range(engine.num_io_tensors):
        context.set_tensor_address(engine.get_tensor_name(i), allocations[i])

    # Create a CUDA stream for asynchronous execution
    err, stream = cudart.cudaStreamCreate()
    check_cuda_error(err)

    # Run inference using the TensorRT engine
    context.execute_async_v3(stream_handle=stream)
    err = cudart.cudaStreamSynchronize(stream)
    check_cuda_error(err)

    # Prepare numpy array for output and copy results from GPU to CPU
    output_shape = outputs[0]["shape"]
    output = np.empty(output_shape, dtype=outputs[0]["dtype"])

    err = cudart.cudaMemcpy(
        output.ctypes.data,
        outputs[0]["allocation"],
        outputs[0]["size"],
        cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost,
    )
    check_cuda_error(err)

    # Free all GPU memory allocations
    for allocation in allocations:
        err = cudart.cudaFree(allocation)
        check_cuda_error(err)

    # Destroy the CUDA stream
    err = cudart.cudaStreamDestroy(stream)
    check_cuda_error(err)

    return output

import onnxruntime as ort
def run_inference_onnx(session, input_data: np.ndarray):
    output = session.run(None, {'x': input_data})[0]
    return output

## 让我们比较一下性能！

我们将多次运行这两个模型，以获得准确的性能对比结果。这将向你展示 TensorRT 带来的基准加速效果。

更多关于如何进一步优化引擎的信息，请参阅 https://docs.nvidia.com/deeplearning/tensorrt/latest/index.html。

In [ ]:
# Create a sample input
sample_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

# Benchmark ONNX Runtime
session = ort.InferenceSession(onnx_path)
onnx_times = []
for _ in range(100):
    start_time = time.time()
    _ = run_inference_onnx(session, sample_input)
    onnx_times.append(time.time() - start_time)

# Benchmark TensorRT
with open(engine_path, "rb") as f, trt.Runtime(logger) as runtime:
    engine = runtime.deserialize_cuda_engine(f.read())
trt_times = []
for _ in range(100):
    start_time = time.time()
    _ = run_inference_trt(engine, sample_input)
    trt_times.append(time.time() - start_time)

print(f"ONNX Runtime Average Time: {np.mean(onnx_times)*1000:.2f} ms")
print(f"TensorRT Average Time: {np.mean(trt_times)*1000:.2f} ms")
print(f"Speedup: {np.mean(onnx_times)/np.mean(trt_times):.2f}x")

## 步骤 4：在真实图像上运行推理

现在让我们在真实图像上测试优化后的模型！我们将：
1. 下载示例图像
2. 加载 ImageNet 类别标签
3. 进行预测并展示结果

这将演示优化后的 TensorRT 引擎在实际场景中的表现。

In [ ]:
# Download a sample image
download_file("https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg", root / "test_image.jpg")

from PIL import Image
from IPython.display import display

# Open and display the image
img = Image.open(root/"test_image.jpg")
display(img)

In [ ]:
def load_imagenet_labels():
    # Download ImageNet labels if not exists
    if not (root / "imagenet_classes.txt").is_file():
        download_file("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt", root / "imagenet_classes.txt")
    # Read the labels
    with open(root / "imagenet_classes.txt") as f:
        categories = [s.strip() for s in f.readlines()]
    return categories

# Load ImageNet labels
categories = load_imagenet_labels()

In [ ]:
# Load and preprocess a test image
test_image_path = root / "test_image.jpg"
input_data = load_and_preprocess_image(test_image_path)

# Run inference
output = run_inference_trt(engine, input_data)

# Get top 5 predictions
top5_idx = np.argsort(output[0])[-5:][::-1]
print("Top 5 predictions:")
for idx in top5_idx:
    print(f"{categories[idx]}: {output[0][idx]:.2f}%")
assert categories[top5_idx[0]] == "Samoyed", 'Incorrect prediction'
print('Correctly recognized!')
print('Notebook executed successfully')

## 恭喜！🎉

### 你已成功完成：
1. 加载预训练的 ONNX 格式 EfficientNet 模型
2. 将其转换为 TensorRT 引擎
3. 实现推理速度的显著提升
4. 对真实图像进行预测
5. 学习如何使用计时缓存加速引擎构建并确保构建结果的确定性

### 后续步骤

既然你已经掌握了 ONNX 到 TensorRT 的工作流程，现在可以：
- 将你自己的 PyTorch/TensorFlow 模型导出为 ONNX 格式
- 尝试 TensorRT 中不同的优化设置
- 将此流程应用于生产模型，借助 NVIDIA GPU 即刻获得性能提升！